# Generalized Wishart process for second-order tensors

GWP receives a second-order tensor coefficient field, not raw dMRI.

## 1. Prepare the Colab environment

In [ ]:
from pathlib import Path
import subprocess
import sys
import zipfile

PROJECT_ROOT = Path('/content/dMRI_Interpolation_Repository')
DATA_ROOT = Path('/content/dMRI_Data')
if not PROJECT_ROOT.is_dir():
    from google.colab import files
    print('Upload dMRI_Interpolation_Repository.zip')
    uploaded = files.upload()
    archives = [name for name in uploaded if name.lower().endswith('.zip')]
    if not archives:
        raise FileNotFoundError('No ZIP archive was uploaded.')
    with zipfile.ZipFile(archives[0]) as archive:
        archive.extractall('/content')
if not PROJECT_ROOT.is_dir():
    raise FileNotFoundError(f'Project directory not found: {PROJECT_ROOT}')
if not (DATA_ROOT / 'TENSOR_2x2x2.nii.gz').is_file():
    from google.colab import files
    print('Upload dMRI_Data.zip')
    uploaded = files.upload()
    archives = [name for name in uploaded if name.lower().endswith('.zip')]
    if not archives:
        raise FileNotFoundError('No dMRI data ZIP archive was uploaded.')
    with zipfile.ZipFile(archives[0]) as archive:
        archive.extractall('/content')
if not (DATA_ROOT / 'TENSOR_2x2x2.nii.gz').is_file():
    raise FileNotFoundError(f'dMRI data folder not found: {DATA_ROOT}')
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q', '-r',
    str(PROJECT_ROOT / 'requirements_colab.txt')
])
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print('Project ready:', PROJECT_ROOT)
print('External dMRI data ready:', DATA_ROOT)

## 2. Load and downsample the tensor field

In [ ]:
import matplotlib.pyplot as plt
from gwp_tensor import gwp_interpolate
from tensor_utils import frobenius_error, load_tensor_field, unsampled_mask

path = PROJECT_ROOT / 'data' / 'tensors' / 'training_data2.mat'
reference = load_tensor_field(path)
low = reference[:, ::2, ::2]
print('Reference:', reference.shape, 'Low resolution:', low.shape)

## 3. GWP interpolation and tensor-domain evaluation

In [ ]:
gwp = gwp_interpolate(low, reference.shape[1:], factor=2, length_scale=2.0)
metrics = frobenius_error(
    gwp['coefficients'], reference, order=2,
    mask=unsampled_mask(reference.shape[1:], 2),
)
print(f"Frobenius distance: {metrics['mean']:.6g} +/- {metrics['std']:.6g}")
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(reference[5], cmap='viridis'); axes[0].set_title('Reference T11')
axes[1].imshow(gwp['coefficients'][5], cmap='viridis'); axes[1].set_title('GWP T11')
axes[2].imshow(metrics['map'], cmap='magma'); axes[2].set_title('Frobenius error')
for ax in axes: ax.axis('off')
plt.tight_layout()